# Assignment 3: Cluster-Analyse

**Matrikelnummer:** 2130238  
**Datensatz:** dataset_cleaned.csv (Bankkunden-Daten)  
**Methode:** k-Means Clustering (unüberwachtes Lernen)

---
- [Task 1: Data Preprocessing](#task1)
- [Task 2: Clustering-Analyse](#task2)
- [Task 3: Cluster-Profiling](#task3)
- [Task 4: Marketing-Empfehlungen](#task4)

## Setup: Bibliotheken, Datensatz & Grundkonfiguration

Alle benötigten Bibliotheken werden einmalig importiert und der bereinigte Datensatz geladen. Da Clustering ein **unsupervised** (unüberwachtes) Verfahren ist, gibt es keine Zielvariable `y` und keinen Train-Test-Split — das Modell erhält keine Labels und sucht eigenständig nach Mustern in den Daten. Es werden alle 8.521 Beobachtungen für die Analyse verwendet.

In [1]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

warnings.filterwarnings('ignore')
plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=12)

# ── Konstante ─────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
COLORS       = [plt.get_cmap('tab10')(i) for i in range(10)]  # Einheitliche Farben fuer alle Plots

# ── Datensatz laden ───────────────────────────────────────────────────────────
df = pd.read_csv('../data/dataset_cleaned.csv')

print(f'Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten')
print(f'Churn-Quote im Datensatz: {df["Churn"].mean()*100:.1f}%')
df.head()


Datensatz geladen: 8521 Zeilen, 24 Spalten
Churn-Quote im Datensatz: 7.3%


,CCreditScore,CAge,CTenure,CBalance,CNumOfProducts,CHasCrCard,CIsActiveMember,CEstimatedSalary,Account_Age_Months,Avg_Monthly_Transactions,...,Is_Holiday_Onboarding,Churn,CLV_Continuous,CBalance_is_negative,CGeography_Germany,CGeography_Spain,CGender_Male,CSeason_Spring,CSeason_Summer,CSeason_Winter
0,553.295,39,2,131008.168572,1,1,0,174570,37,8,...,0,0,21696.256022,0,0,0,1,0,1,0
1,552.939,33,4,118529.788812,1,0,0,95123,24,11,...,0,0,15918.891128,0,0,0,1,0,0,0
2,688.746,40,1,-879.724555,2,1,1,145256,1,13,...,0,0,10361.315773,1,0,1,1,0,0,1
3,771.941,40,9,125563.132706,1,1,0,71149,2,12,...,1,0,15189.776245,0,0,0,1,0,0,1
4,484.093,55,5,95759.918567,1,0,1,77539,24,8,...,0,0,14631.814966,0,0,0,1,0,0,1


<a id='task1'></a>

## Task 1: Data Preprocessing

Beim Clustering ist das Preprocessing besonders kritisch, weil der k-Means-Algorithmus ausschließlich auf **euklidischen Abständen** (geometrischer Abstand zwischen zwei Punkten im Merkmalsraum) basiert. Features mit großen Wertebereichen — z. B. `CBalance` bis 220.000 € vs. `CNumOfProducts` bis 4 — würden die Distanzberechnung und damit das Clustering-Ergebnis dominieren, wenn sie nicht skaliert werden.

### 1.1 Feature-Auswahl

Für die Cluster-Analyse werden **verhaltens- und wertbezogene Features** ausgewählt. Folgende Überlegungen standen dabei im Vordergrund:

- **Einbezogen:** Kontinuierliche und ordinale Merkmale, die direkt das Kundenverhalten oder den Kundenwert beschreiben — darunter `CBalance`, `CLV_Continuous`, `CIsActiveMember` und `Churn`. Das `Churn`-Merkmal wird bewusst als **Verhaltensindikator** einbezogen, nicht als Zielvariable: Da es sich um eine abgeschlossene historische Beobachtung handelt, erlaubt es, abgewanderte Kunden als eigenständiges Segment zu identifizieren und gezielt zu adressieren.
- **Ausgeschlossen:** Rein binäre Dummy-Variablen ohne inhaltliche Ordnung (`CGeography_*`, `CSeason_*`, `CHasCrCard`), da diese die Distanzberechnung verzerren und kaum zur Clusterstruktur beitragen.

In [2]:
# ── Feature-Auswahl für Clustering ──────────────────────────────────────────
X_raw = [
    'CCreditScore',              # Kreditwürdigkeit
    'CAge',                      # Alter
    'CBalance',                  # Kontostand
    'CNumOfProducts',            # Anzahl Bankprodukte
    'CEstimatedSalary',          # Geschätztes Jahresgehalt
    'CLV_Continuous',            # Customer Lifetime Value
    'Avg_Monthly_Transactions',  # Durchschnittliche monatliche Transaktionen
    'Mobile_App_Usage_Hours',    # App-Nutzung in Stunden (digitales Engagement)
    'Support_Tickets_Count',     # Anzahl Support-Anfragen
    'CIsActiveMember',           # Aktives Mitglied (0/1)
    'Churn',                     # Churn-Status (0/1 — als Verhaltensmerkmal)
]

X = df[X_raw].copy()

print(f'Ausgewählte Features: {len(X_raw)}')
print(f'Beobachtungen für Clustering: {len(X)}')
print()
print('Deskriptive Statistik der ausgewählten Features:')
print(X.describe().round(2))

Ausgewählte Features: 11
Beobachtungen für Clustering: 8521

Deskriptive Statistik der ausgewählten Features:
       CCreditScore     CAge   CBalance  CNumOfProducts  CEstimatedSalary  \
count       8521.00  8521.00    8521.00         8521.00           8521.00   
mean         651.59    37.99   74032.94            1.54         100747.37   
std           95.70    10.37   62580.47            0.53          54594.69   
min          402.12    18.00   -7218.83            1.00              2.00   
25%          585.09    31.00     758.79            1.00          56999.00   
50%          653.11    36.00   93650.83            2.00          99991.00   
75%          717.71    42.00  126615.63            2.00         144763.00   
max          857.46    84.00  220214.29            4.00         204659.00   

       CLV_Continuous  Avg_Monthly_Transactions  Mobile_App_Usage_Hours  \
count         8521.00                    8521.0                 8521.00   
mean         13674.75                      11.

### 1.2 Feature Scaling mit StandardScaler

Der **StandardScaler** transformiert jedes Feature auf Mittelwert 0 und Standardabweichung 1 (z-Score-Normalisierung):

$$z = \frac{x - \mu}{\sigma}$$

Dadurch werden alle Features gleichgewichtig behandelt und kein Merkmal dominiert die Distanzberechnung aufgrund seiner Skalierungsgröße. Dies entspricht der Anforderung aus der Vorlesung (Kap. 4: Scale Issues).

In [3]:
# ── StandardScaler anwenden ──────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Validierung der Skalierung
print('=== Validierung: Skalierung erfolgreich? ===')
means = X_scaled_df.mean()
stds  = X_scaled_df.std()
print(f'Max. Abweichung Mittelwert von 0: {means.abs().max():.6f}  ->  OK')
print(f'Std-Abweichung: Min={stds.min():.4f}, Max={stds.max():.4f}  ->  OK')
print()
print('Skalierte Daten — erste 3 Zeilen:')
print(X_scaled_df.head(3).round(3))

=== Validierung: Skalierung erfolgreich? ===
Max. Abweichung Mittelwert von 0: 0.000000  ->  OK
Std-Abweichung: Min=1.0001, Max=1.0001  ->  OK

Skalierte Daten — erste 3 Zeilen:
   CCreditScore   CAge  CBalance  CNumOfProducts  CEstimatedSalary  \
0        -1.027  0.097     0.910          -1.007             1.352   
1        -1.031 -0.481     0.711          -1.007            -0.103   
2         0.388  0.194    -1.197           0.874             0.815   

   CLV_Continuous  Avg_Monthly_Transactions  Mobile_App_Usage_Hours  \
0           1.472                    -1.182                   0.587   
1           0.412                    -0.273                   1.295   
2          -0.608                     0.334                  -0.330   

   Support_Tickets_Count  CIsActiveMember  Churn  
0                  0.638           -1.089 -0.281  
1                 -0.666           -1.089 -0.281  
2                 -0.666            0.919 -0.281  


**Ergebnis Preprocessing:** Die Skalierung ist erfolgreich — alle Features haben Mittelwert ~0 und Standardabweichung ~1 und sind damit gleichberechtigt in die Distanzberechnung einbezogen. Der Datensatz mit 8.521 Beobachtungen und 11 Features ist bereit für die Clustering-Analyse.

<a id='task2'></a>

## Task 2: Clustering-Analyse

### 2.1 Algorithmus: k-Means Clustering

**Was ist k-Means?**  
k-Means ist ein unsupervised Clustering-Algorithmus, der Beobachtungen in *k* disjunkte Gruppen (Cluster) aufteilt. Das Ziel ist, die **Intra-Cluster-Ähnlichkeit** (Kohäsion innerhalb eines Clusters) zu maximieren und die **Inter-Cluster-Distanz** (Separation zwischen Clustern) zu minimieren.

**Funktionsweise:**
1. *k* Centroids (Clusterschwerpunkte) zufällig initialisieren
2. Jede Beobachtung dem nächsten Centroid zuweisen (Euclidean Distance)
3. Centroids neu berechnen als Mittelwert der zugewiesenen Punkte
4. Schritte 2 und 3 wiederholen, bis keine Änderung mehr eintritt (lokales Optimum)

**Warum k-Means?** k-Means eignet sich sehr gut für große Datensätze mit flacher Geometrie und überwiegend numerischen Features — genau der vorliegende Fall mit 8.521 Beobachtungen. Der Algorithmus ist gut skalierbar und die Ergebnisse sind direkt interpretierbar.

### 2.2 Optimale Clusteranzahl: Elbow-Methode und Silhouette-Score

Da *k* bei k-Means vorab angegeben werden muss, werden zwei komplementäre Methoden eingesetzt:

**Elbow-Methode (Inertia / WCSS):**  
Inertia = Within-Cluster Sum of Squares = Summe der quadratischen Abstände aller Punkte zu ihrem Centroid. Niedrige Inertia bedeutet kompaktere Cluster. Da Inertia mit jedem weiteren Cluster automatisch sinkt, sucht man den Knick (Elbow), ab dem der Rückgang deutlich geringer wird.

**Silhouette-Score:**  
Bewertet, wie gut jede Beobachtung zum eigenen Cluster passt im Vergleich zum nächstbesten Cluster. Wertebereich: −1 bis +1.  
- Wert nahe **+1**: Beobachtung passt klar zum zugewiesenen Cluster  
- Wert nahe **0**: Beobachtung liegt auf der Grenze zweier Cluster  
- Wert nahe **−1**: Beobachtung passt besser ins Nachbar-Cluster

In [ ]:
# ── Elbow und Silhouette für k = 2 bis 10 ───────────────────────────────────
K_RANGE         = range(2, 11)
inertia_list    = []
silhouette_list = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertia_list.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    silhouette_list.append(sil)
    print(f'k={k:2d} | Inertia: {km.inertia_:>10.0f} | Silhouette: {sil:.4f}')

In [ ]:
# ── Visualisierung: Elbow-Plot und Silhouette-Plot ───────────────────────────
best_k_idx = int(np.argmax(silhouette_list))
best_k     = list(K_RANGE)[best_k_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow-Plot
ax1.plot(list(K_RANGE), inertia_list, 'o-', color='steelblue', linewidth=2, markersize=7)
ax1.axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'k={best_k} (gewählt)')
ax1.set_xlabel('Anzahl Cluster (k)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow-Methode')
ax1.legend()
ax1.grid(True, alpha=0.4)
ax1.set_xticks(list(K_RANGE))

# Silhouette-Plot als Balkendiagramm
ax2.bar(list(K_RANGE), silhouette_list,
        color=['tomato' if k == best_k else 'steelblue' for k in K_RANGE],
        edgecolor='black', linewidth=0.5, alpha=0.85)
ax2.set_xlabel('Anzahl Cluster (k)')
ax2.set_ylabel('Silhouette-Score')
ax2.set_title('Silhouette-Score je k')
ax2.grid(axis='y', alpha=0.4)
ax2.set_xticks(list(K_RANGE))
for i, (k, s) in enumerate(zip(K_RANGE, silhouette_list)):
    ax2.text(k, s + 0.001, f'{s:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Bestimmung der optimalen Clusteranzahl', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Gewähltes k = {best_k} (höchster Silhouette-Score: {max(silhouette_list):.4f})')

**Interpretation Elbow & Silhouette:**

Der Silhouette-Score ist bei **k = 3** mit 0,1808 am höchsten — dies ist das optimale *k*. Der Elbow-Plot bestätigt diesen Befund: Die Inertia sinkt von k=2 zu k=3 noch deutlich (79.182 → 70.944), fällt danach aber nur noch moderat. Der Gesamtwert des Silhouette-Scores von ~0,18 ist relativ niedrig. Das ist bei realen, hochdimensionalen Kundendatensätzen typisch und kein Fehler: Echte Kundensegmente sind in der Regel nicht klar trennbar — die Grenzlinien zwischen Gruppen sind fließend, nicht hart. Entscheidend ist, dass k=3 im Vergleich aller getesteten Werte die beste Separation liefert und inhaltlich interpretierbare, klar unterschiedliche Kundengruppen erzeugt.

In [ ]:
# ── Finales k-Means Modell trainieren ────────────────────────────────────────
OPTIMAL_K    = best_k
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
kmeans_final.fit(X_scaled)
df['Cluster'] = kmeans_final.labels_

print(f'=== Finales Modell: k-Means mit k={OPTIMAL_K} ===')
print(f'Inertia (WCSS): {kmeans_final.inertia_:,.2f}')
print()
print('Cluster-Verteilung:')
for c in range(OPTIMAL_K):
    n = (df['Cluster'] == c).sum()
    print(f'  Cluster {c}: {n:>5} Kunden ({n/len(df)*100:.1f}%)')

In [ ]:
# ── Silhouette-Analyse für das finale Modell ─────────────────────────────────
sil_score_final = silhouette_score(X_scaled, kmeans_final.labels_)
sil_samples     = silhouette_samples(X_scaled, kmeans_final.labels_)

print(f'=== Clusterqualität: k={OPTIMAL_K} ===')
print(f'Silhouette-Score (gesamt) : {sil_score_final:.4f}')
print(f'Inertia                   : {kmeans_final.inertia_:,.2f}')
print()
print('Silhouette-Score je Cluster:')
for c in range(OPTIMAL_K):
    mask  = kmeans_final.labels_ == c
    sc    = sil_samples[mask].mean()
    n     = mask.sum()
    neg_n = (sil_samples[mask] < 0).sum()
    print(f'  Cluster {c}: {sc:.4f}  (n={n:>5}, negative Werte: {neg_n} = {neg_n/n*100:.1f}%)')

**Interpretation Silhouette-Analyse:**

Cluster 0 erzielt mit 0,2164 den höchsten Silhouette-Score — dieses Segment ist am klarsten vom Rest abgegrenzt, was sich durch seinen besonderen Charakter als reines Churn-Segment erklärt (alle Kunden dieses Clusters haben `Churn = 1`). Cluster 1 (0,1701) und Cluster 2 (0,1891) überschneiden sich in einigen Dimensionen, sind aber über den Kontostand (`CBalance`) und den Customer Lifetime Value (`CLV_Continuous`) klar differenzierbar. Die negativen Silhouette-Werte je Cluster (Beobachtungen auf der Grenze zweier Cluster) sind mit unter 20 % in einem akzeptablen Bereich.

In [ ]:
# ── PCA-Visualisierung: Cluster im 2D-Raum ───────────────────────────────────
# PCA (Principal Component Analysis = Hauptkomponentenanalyse):
# Reduziert die 11-dimensionalen Daten auf 2 Dimensionen zur Visualisierung.
# Die ersten zwei Hauptkomponenten erklären den größten Varianzanteil.

pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(10, 6))
for c in range(OPTIMAL_K):
    mask = kmeans_final.labels_ == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               s=15, alpha=0.45, color=COLORS[c],
               label=f'Cluster {c} (n={mask.sum()})')

centroids_pca = pca.transform(kmeans_final.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           s=220, marker='X', c='black', zorder=6, label='Centroids')
for c in range(OPTIMAL_K):
    ax.annotate(f'C{c}', (centroids_pca[c, 0], centroids_pca[c, 1]),
                textcoords='offset points', xytext=(6, 5),
                fontsize=11, fontweight='bold')

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% erklärte Varianz)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% erklärte Varianz)')
ax.set_title(f'k-Means Cluster (k={OPTIMAL_K}) — PCA-Visualisierung')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'PC1={explained[0]*100:.1f}%, PC2={explained[1]*100:.1f}%, Gesamt={sum(explained)*100:.1f}%')

**Interpretation PCA-Plot:**

Der PCA-Plot projiziert die 11-dimensionalen Daten auf die zwei Dimensionen mit dem größten Informationsgehalt. Cluster 0 (abgewanderte Kunden) ist klar nach rechts verschoben und vom Rest abgegrenzt — das bestätigt den hohen Silhouette-Score dieses Segments. Cluster 1 und 2 überlappen im 2D-Raum stärker, was der niedrigeren Separation zwischen diesen Gruppen entspricht. Da PC1 und PC2 zusammen nur einen Teil der Gesamtvarianz erklären, ist die sichtbare Überlappung kein Widerspruch zur tatsächlichen Trennqualität im vollen 11-dimensionalen Raum.

In [ ]:
# ── Voronoi-Diagramm: Clusterpartitionierung visualisieren ──────────────────
# Das Voronoi-Diagramm zeigt die Entscheidungsgrenzen zwischen Clustern:
# Jede Region gehört zum nächstgelegenen Centroid (k-Means Entscheidungsregel).
# Dies visualisiert exakt, wie der k-Means Algorithmus den Raum partitioniert.

from scipy.spatial import Voronoi, voronoi_plot_2d

# Bezugskerne (Centroids im PCA-Raum)
centroids_pca = pca.transform(kmeans_final.cluster_centers_)

# Voronoi-Diagramm berechnen
vor = Voronoi(centroids_pca)

fig, ax = plt.subplots(figsize=(12, 7))

# Begrenzungsbox für Voronoi-Plot definieren
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# Voronoi-Regionen zeichnen (Partitionierungslinien)
voronoi_plot_2d(vor, ax=ax, show_points=False, show_vertices=False, 
                line_colors='gray', line_width=1.5, line_alpha=0.6)

# Clusterpunkte überlagern, farbcodiert nach Cluster
for c in range(OPTIMAL_K):
    mask = kmeans_final.labels_ == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               s=20, alpha=0.5, color=COLORS[c],
               label=f'Cluster {c} (n={mask.sum()})', edgecolors='none')

# Centroids deutlich markieren
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           s=300, marker='*', c='black', edgecolors='white', linewidth=2,
           zorder=10, label='Centroids')

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% Varianz)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% Varianz)')
ax.set_title(f'Voronoi-Diagramm: k-Means Cluster-Partitionierung (k={OPTIMAL_K})\n(Grau: Entscheidungsgrenzen zwischen Clustern)')
ax.legend(loc='upper right', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

**Interpretation Voronoi-Diagramm:**

Das Voronoi-Diagramm zeigt die **exakte Partitionierung des Feature-Raums** durch k-Means. Jede farbige Region entspricht allen Punkten, die demselben Centroid (schwarzer Stern) am nächsten liegen — das ist genau die k-Means Entscheidungsregel.

- **Graue Linien:** Entscheidungsgrenzen zwischen Clustern. An diesen Grenzen sind zwei oder mehr Centroids gleich weit entfernt.
- **Cluster 0 (blau):** Kleine, räumlich isolierte Region oben rechts — zeigt klar, dass abgewanderte Kunden sich von den anderen deutlich unterscheiden (hohe Separation).
- **Cluster 1 & 2 (orange/grün):** Größere, ineinander verschlungene Regionen — bestätigt die niedrigere Silhouette-Separation zwischen Treue-Kunden, da die Grenzen weniger klar sind.
- **Größe der Regionen:** Spiegelt die relative Cluster-Größe wider (C1 am größten, C0 am kleinsten im PCA-Raum).

Das Voronoi-Diagramm verdeutlicht visuell, **warum k=3 optimal ist**: Cluster 0 ist natürlich und deutlich abgegrenzt, während die Separation zwischen C1 und C2 moderater ist — genau wie der Silhouette-Score in Abschnitt 2.2 zeigte.

<a id='task3'></a>

## Task 3: Cluster-Profiling

Die drei Cluster werden nun inhaltlich charakterisiert. Für jede Gruppe werden die typischen Kundenmerkmale herausgearbeitet, Unterschiede und Gemeinsamkeiten identifiziert und die Ergebnisse visualisiert.

### 3.1 Deskriptive Statistiken je Cluster

In [ ]:
# ── Profil-Mittelwerte je Cluster ────────────────────────────────────────────
PROFILE_FEATURES = [
    'CCreditScore', 'CAge', 'CBalance', 'CNumOfProducts',
    'CEstimatedSalary', 'CLV_Continuous', 'Avg_Monthly_Transactions',
    'Mobile_App_Usage_Hours', 'Support_Tickets_Count',
    'CIsActiveMember', 'Churn'
]

cluster_profile = df.groupby('Cluster')[PROFILE_FEATURES].mean().round(2)
cluster_sizes   = df.groupby('Cluster').size().rename('n_Kunden')

print('=== Cluster-Profile: Mittelwerte je Cluster ===')
print(pd.concat([cluster_sizes, cluster_profile], axis=1).to_string())

### 3.2 Cluster-Beschreibung und Profilnamen

Auf Basis der berechneten Mittelwerte lassen sich drei klar unterscheidbare Kundenprofile ableiten:

| Cluster | Profilname | n | Churn | CLV | Kontostand | Aktiv |
|---------|-----------|---|-------|-----|------------|-------|
| **0** | Abgewanderte Kunden | 625 (7,3%) | 100% | 13.779 € | 88.515 € | 37,8% |
| **1** | Treue Premium-Kunden | 4.619 (54,2%) | 0% | 17.464 € | 121.878 € | 56,6% |
| **2** | Treue Standard-Kunden | 3.277 (38,5%) | 0% | 8.314 € | 3.832 € | 54,0% |

**Cluster 0 — Abgewanderte Kunden (625 Kunden, 7,3%):**  
Dieses Cluster enthält ausschließlich abgewanderte Kunden (`Churn = 1`). Sie sind mit durchschnittlich 44,7 Jahren merklich älter als die anderen Gruppen und weisen trotz eines soliden Kontostands von 88.515 € eine niedrige Aktivitätsrate von nur 37,8% auf. Der CLV liegt mit 13.779 € im mittleren Bereich — ein Zeichen, dass diese Kunden durchaus wertvoll waren, aber verloren gegangen sind.

**Cluster 1 — Treue Premium-Kunden (4.619 Kunden, 54,2%):**  
Die größte und wertvollste Gruppe mit einem durchschnittlichen CLV von 17.464 € und dem höchsten Kontostand von 121.878 €. Keine Abwanderung, überdurchschnittliche Aktivitätsrate von 56,6%. Dies ist das Kern-Segment der Bank — finanzstark, engagiert und loyal.

**Cluster 2 — Treue Standard-Kunden (3.277 Kunden, 38,5%):**  
Ebenfalls kein Churn, aber deutlich geringerer Kontostand (3.832 €) und CLV (8.314 €). Das Alter ist mit 37,5 Jahren identisch zu Cluster 1, die Aktivitätsrate ähnlich (54,0%). Das Hauptunterscheidungsmerkmal ist die Kontosaldo-Höhe: Diese Kunden halten kaum Guthaben bei der Bank — entweder Neukunden oder Kunden, die ihr Geld anderweitig verwalten.

### 3.3 Vergleichende Visualisierungen

In [ ]:
# ── Bar-Vergleich: 8 zentrale Features je Cluster ────────────────────────────
compare_features = [
    'CCreditScore', 'CAge', 'CBalance', 'CEstimatedSalary',
    'CLV_Continuous', 'Avg_Monthly_Transactions',
    'Mobile_App_Usage_Hours', 'CIsActiveMember'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(compare_features):
    vals = cluster_profile[feat]
    bars = axes[i].bar(
        [f'C{c}' for c in vals.index], vals.values,
        color=[COLORS[c] for c in vals.index], edgecolor='black', linewidth=0.5
    )
    gm = df[feat].mean()
    axes[i].axhline(gm, color='gray', linestyle='--', linewidth=1.2,
                    alpha=0.7, label=f'Ø {gm:.1f}')
    axes[i].set_title(feat.replace('C', '', 1).replace('_', ' '))
    axes[i].set_ylabel('Mittelwert')
    axes[i].legend(fontsize=8)
    axes[i].grid(axis='y', alpha=0.4)

    # Fester Headroom: 15% über dem Maximum (Balken oder Referenzlinie)
    y_top = max(vals.values.max(), gm) * 1.15
    axes[i].set_ylim(0, y_top)

    for bar, val in zip(bars, vals.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, val + y_top * 0.01,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle(f'Feature-Mittelwerte je Cluster vs. Gesamtdurchschnitt (k={OPTIMAL_K})',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Der Vergleich der Feature-Mittelwerte bestätigt die Cluster-Profile aus Abschnitt 3.2 und macht die Unterschiede visuell greifbar:

- **CreditScore & EstimatedSalary & Avg. Monthly Transactions & Mobile App Usage:** Alle drei Cluster liegen nahezu auf der Referenzlinie (Gesamtdurchschnitt). Diese Features differenzieren die Segmente nicht — sie sind geteilte Eigenschaften aller Kunden.

- **Age:** Cluster 0 liegt mit 44,7 Jahren deutlich über dem Durchschnitt (38,0 Jahre), Cluster 1 und 2 liegen identisch darunter (37,5 Jahre). Das erhöhte Alter ist ein spezifisches Merkmal abgewanderter Kunden.

- **Balance:** Das stärkste visuelle Differenzierungsmerkmal. Cluster 1 übertrifft den Gesamtdurchschnitt um ca. +64% (121.878 € vs. 74.033 €), während Cluster 2 mit 3.832 € weit darunter liegt (−95%). Cluster 0 liegt mit 88.515 € ebenfalls überdurchschnittlich — abgewanderte Kunden hatten somit ein solides Guthaben.

- **CLV_Continuous:** Spiegelt das Balance-Muster wider: Cluster 1 liegt mit 17.464 € klar über dem Durchschnitt (13.675 €), Cluster 2 deutlich darunter (8.314 €). Cluster 0 liegt nahe am Durchschnitt (13.779 €).

- **IsActiveMember:** Cluster 0 liegt mit 0,4 (40%) unter dem Durchschnitt (0,5), Cluster 1 mit 0,6 (60%) darüber. Die geringe Aktivitätsrate der abgewanderten Kunden ist ein retrospektiver Frühindikator für Abwanderung.

In [ ]:
# ── Violin-Plots: Verteilung zentraler Features je Cluster ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat, label in [
    (axes[0], 'CAge',           'Alter (Jahre)'),
    (axes[1], 'CBalance',       'Kontostand (€)'),
    (axes[2], 'CLV_Continuous', 'CLV (€)')
]:
    data = [df[df['Cluster'] == c][feat].values for c in range(OPTIMAL_K)]
    parts = ax.violinplot(data, positions=range(OPTIMAL_K), showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(COLORS[i])
        pc.set_alpha(0.7)
    ax.set_xticks(range(OPTIMAL_K))
    ax.set_xticklabels([f'Cluster {c}' for c in range(OPTIMAL_K)])
    ax.set_ylabel(label)
    ax.set_title(f'Verteilung: {label} je Cluster')
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Verteilungen zentraler Features je Cluster', fontsize=13)
plt.tight_layout()
plt.show()


**Alter:** Cluster 0 zeigt eine deutlich breitere und gleichmäßigere Verteilung (20–84 Jahre) als Cluster 1 und 2, die eng um ~35–38 Jahre konzentriert sind. Der höhere Mittelwert aus dem Bar-Chart spiegelt also nicht nur ein höheres Durchschnittsalter wider, sondern eine insgesamt heterogenere Altersstruktur der abgewanderten Kunden.

**Kontostand:** Der auffälligste Befund. Cluster 2 ist extrem komprimiert nahe 0 € — kaum Streuung, fast alle Kunden dieses Segments halten einen nahezu identisch niedrigen Kontostand. Der niedrige Mittelwert aus dem Bar-Chart (3.832 €) ist also kein Ausreißereffekt, sondern strukturell. Cluster 1 und Cluster 0 sind hingegen breit verteilt (bis ~220.000 €), wobei Cluster 1 eine höhere Dichte im oberen Bereich aufweist.

**CLV:** Cluster 1 hat die breiteste Verteilung mit dem höchsten Median — hier gibt es sowohl Standard- als auch Hochwertkunden. Cluster 2 ist eng im unteren Bereich konzentriert (wenig Streuung, konsistent niedriger CLV), was die −39% Abweichung vom Durchschnitt aus dem Bar-Chart bestätigt. Cluster 0 liegt dazwischen mit moderater Breite.

In [ ]:
# ── Heatmap: Normalisierte Feature-Mittelwerte je Cluster ────────────────────
# Farbe = Min-Max-normalisiert je Feature (0 = niedrigster Cluster, 1 = höchster Cluster)
# Zellinhalt = Rohwert des jeweiligen Cluster-Mittelwerts
heatmap_features = [
    feat for feat in X.columns
    if feat not in ['CNumOfProducts', 'CIsActiveMember', 'Churn']
]

CLUSTER_NAMES = {
    0: 'C0: Abgewandert',
    1: 'C1: Premium-Kunden',
    2: 'C2: Standard-Kunden'
}

profile_heat = cluster_profile[heatmap_features].copy()
profile_norm = (profile_heat - profile_heat.min()) / (profile_heat.max() - profile_heat.min())

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(profile_norm.values, aspect='auto', cmap='RdYlGn')

ax.set_xticks(range(len(heatmap_features)))
ax.set_xticklabels([f.replace('C', '', 1).replace('_', ' ') for f in heatmap_features],
                   rotation=35, ha='right')
ax.set_yticks(range(OPTIMAL_K))
ax.set_yticklabels([CLUSTER_NAMES[c] for c in range(OPTIMAL_K)])

for i in range(OPTIMAL_K):
    for j in range(len(heatmap_features)):
        raw = profile_heat.iloc[i, j]
        ax.text(j, i, f'{raw:.1f}', ha='center', va='center', fontsize=9)

plt.colorbar(im, ax=ax, label='Normalisierter Wert (0 = niedrig, 1 = hoch)')
ax.set_title('Cluster-Heatmap: Feature-Mittelwerte (Rohwerte in Zellen, Farbe normalisiert)')
plt.tight_layout()
plt.show()


**Interpretation Heatmap:**

Die Heatmap verdichtet die Clusterprofile auf einen Blick: Grün markiert jeweils die relativ höchsten, Rot die relativ niedrigsten Werte eines Features zwischen den drei Clustern.

- **Balance** und **CLV** sind die stärksten Trennmerkmale: Cluster 1 ist hier klar am höchsten, Cluster 2 klar am niedrigsten.
- **Age** hebt vor allem Cluster 0 hervor, das im Vergleich zu den loyalen Clustern älter ist.
- **EstimatedSalary**, **CreditScore**, **Avg Monthly Transactions** und **Mobile App Usage Hours** unterscheiden sich dagegen nur gering und tragen wenig zur inhaltlichen Trennung der Cluster bei.

Damit bestätigt die Heatmap die bisherigen Befunde aus Balken- und Violin-Plots in kompakter Form: Cluster 1 ist das wertstarke Premium-Segment, Cluster 2 das saldo- und CLV-schwache Standard-Segment, und Cluster 0 unterscheidet sich vor allem über Alter und Churn-Nähe.




### 3.4 Unterschiede und Gemeinsamkeiten der Cluster

**Gemeinsamkeiten aller Cluster:**

Trotz der klaren Segmentierung teilen alle drei Cluster mehrere grundlegende Eigenschaften:

- **Identisches Einkommensniveau:** Der geschätzte Jahreslohn (`CEstimatedSalary`) liegt in allen Clustern bei ca. 97.000–103.000 € (Abweichung < 4% vom Gesamtdurchschnitt). Das Gehalt ist kein Differenzierungsmerkmal.
- **Vergleichbarer Credit Score:** Der `CCreditScore` variiert kaum zwischen den Clustern (649–653 Punkte), alle liegen im gleichen Kreditwürdigkeitsbereich.
- **Gleiche Transaktionsfrequenz:** Die mittlere Anzahl monatlicher Transaktionen (`Avg_Monthly_Transactions`) ist mit ~11,9 in allen Clustern nahezu identisch.
- **Ähnliche App-Nutzung und Support-Nutzung:** Mobile App-Nutzung (~5 h) und Support-Tickets (~0,5) zeigen kaum Abweichungen zwischen den Gruppen.

**Zentrale Unterschiede:**

Die eigentlichen Differenzierungsmerkmale zwischen den Clustern sind klar konzentriert:

- **Churn** ist mit über 1.200% Abweichung das stärkste Trennmerkmal — Cluster 0 besteht ausschließlich aus abgewanderten Kunden (100%), während Cluster 1 und 2 keine Abwanderung zeigen.
- **Kontostand (`CBalance`)** trennt Cluster 1 und 2 fundamental: Cluster 1 hält durchschnittlich 121.878 €, Cluster 2 nur 3.832 € — eine Abweichung von ca. 95% vom Gesamtmittelwert.
- **Customer Lifetime Value (`CLV_Continuous`)** folgt dem Kontostand-Muster: 17.464 € (C1) vs. 8.314 € (C2) vs. 13.779 € (C0).
- **Alter (`CAge`)** ist nur zwischen Cluster 0 (44,7 Jahre) und den anderen Clustern (je 37,5 Jahre) unterschiedlich — abgewanderte Kunden sind im Schnitt 7 Jahre älter.

In [ ]:
# ── Quantitative Übersicht: Variation je Feature ──────────────────────────────
print('=== Variation je Feature (Max-Abweichung vom Gesamtdurchschnitt in %) ===')
print(f'  {"Feature":<30} {"Abweichung":<14} {"Einordnung"}')
print('  ' + '-'*65)

diff_summary = []
for feat in PROFILE_FEATURES:
    gm = df[feat].mean()
    if gm == 0:
        continue
    dev = (cluster_profile[feat] - gm).abs().max() / abs(gm) * 100
    diff_summary.append((feat, dev, cluster_profile[feat].min(), cluster_profile[feat].max()))
diff_summary.sort(key=lambda x: -x[1])

for feat, dev, mn, mx in diff_summary:
    if dev > 30:
        label = 'Starkes Differenzierungsmerkmal'
    elif dev < 10:
        label = 'Gemeinsamkeit aller Cluster'
    else:
        label = 'Mittlere Variation'
    print(f'  {feat:<30} {dev:>8.1f}%     {label}')

<a id='task4'></a>

## Task 4: Marketing-Empfehlungen

Auf Basis der Cluster-Profile werden konkrete, umsetzbare Marketing-Strategien abgeleitet. Die Empfehlungen sind direkt an die drei identifizierten Segmente geknüpft und basieren auf den tatsächlichen Profilwerten aus Task 3.

In [ ]:
# ── Strategische Positionierungsmatrix: CLV vs. Churn-Rate ───────────────────
fig, ax = plt.subplots(figsize=(9, 6))

clv_global   = df['CLV_Continuous'].mean()
churn_global = df['Churn'].mean() * 100

labels = {0: 'C0: Abgewandert', 1: 'C1: Premium', 2: 'C2: Standard'}
annotation_offsets = {0: (10, 4), 1: (-10, 4), 2: (10, 4)}
annotation_align   = {0: 'left', 1: 'right', 2: 'left'}

for c in range(OPTIMAL_K):
    grp      = df[df['Cluster'] == c]
    clv_m    = grp['CLV_Continuous'].mean()
    churn_p  = grp['Churn'].mean() * 100
    n        = len(grp)
    ax.scatter(clv_m, churn_p, s=n / 4, color=COLORS[c], alpha=0.85,
               edgecolors='black', linewidth=1.2, zorder=5)
    ax.annotate(
        f'C{c}',
        (clv_m, churn_p),
        textcoords='offset points',
        xytext=annotation_offsets[c],
        ha=annotation_align[c],
        fontsize=12,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.8)
    )

ax.axhline(churn_global, color='gray', linestyle='--', alpha=0.6)
ax.axvline(clv_global, color='gray', linestyle=':', alpha=0.6)

cluster_handles = [
    Line2D([0], [0], marker='o', color='w', label=labels[c],
           markerfacecolor=COLORS[c], markeredgecolor='black', markersize=10)
    for c in range(OPTIMAL_K)
]
reference_handles = [
    Line2D([0], [0], color='gray', linestyle='--', label=f'Ø Churn: {churn_global:.1f}%'),
    Line2D([0], [0], color='gray', linestyle=':', label=f'Ø CLV: {clv_global:.0f} €')
]
ax.legend(handles=cluster_handles + reference_handles, loc='upper right', fontsize=11)

ax.set_xlabel('Ø Customer Lifetime Value — CLV (€)')
ax.set_ylabel('Churn-Rate (%)')
ax.set_title('Strategische Cluster-Matrix: CLV vs. Churn-Rate\n(Punktgröße proportional zur Cluster-Größe)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4.1 Segment-spezifische Handlungsempfehlungen

---

**Cluster 0 — Abgewanderte Kunden (625 Kunden | Churn: 100% | CLV: 13.779 €)**  
**Strategie: Rückgewinnung & Analyse**

Da diese Kunden bereits abgewandert sind, steht die Analyse der Abwanderungsursachen und eine gezielte Rückgewinnungskampagne im Vordergrund. Der relativ hohe Kontostand von 88.515 € und der mittlere CLV zeigen, dass es sich um wertvolle Kunden handelt, deren Rückgewinnung lohnenswert ist. Hinzu kommt das Alter (Ø 44,7 Jahre): Diese Zielgruppe bevorzugt persönliche Beratung über traditionelle Kanäle.

- **Rückgewinnungs-Outreach:** Direktes persönliches Kontaktangebot per Telefon oder Brief mit konkreten Mehrwert-Angeboten (z. B. verbesserter Kontozins, Gebührenerlass).
- **Ursachenanalyse:** Exit-Befragungen oder CRM-Auswertungen, um Abwanderungsgründe zu verstehen und zukünftige Churn-Prävention zu verbessern.
- **Frühwarnsystem:** Modell aus Assignment 1 (Churn-Klassifikation) aktiv einsetzen, um ähnliche Risikoprofile bei verbleibenden Kunden frühzeitig zu erkennen.

---

**Cluster 1 — Treue Premium-Kunden (4.619 Kunden | Churn: 0% | CLV: 17.464 €)**  
**Strategie: Retention & Reward (Beste Kunden halten und ausbauen)**

Dieses Segment ist das Rückgrat des Portfolios — größte Gruppe, höchster CLV, kein Churn. Der hohe Kontostand von 121.878 € bietet Potenzial für weitere Anlage- und Kreditprodukte. Ziel ist die langfristige Bindung und schrittweise Wertsteigerung.

- **Loyalitätsprogramm:** Prämienpunkte für Transaktionen und Produktnutzung, einlösbar gegen Cashback oder Zinsboni.
- **Premium-Produktangebote:** Persönliche Finanzberatung, bevorzugte Kreditkonditionen, exklusive Anlageprodukte (z. B. Fonds, ETF-Sparpläne).
- **VIP-Kommunikation:** Dedizierter Kundenberater, Early-Access zu neuen Produkten, Einladungen zu Events.

---

**Cluster 2 — Treue Standard-Kunden (3.277 Kunden | Churn: 0% | CLV: 8.314 €)**  
**Strategie: Aktivierung & Cross-Selling (Potenzial heben)**

Kein Churn, aber deutlich geringerer Kontostand (3.832 €) und CLV (8.314 €) als Cluster 1. Alter und Aktivitätsrate sind fast identisch zu Cluster 1 — der Unterschied liegt nicht im Engagement, sondern im finanziellen Volumen. Das Potenzial zur Entwicklung Richtung Cluster 1 ist groß, wenn der Kontoaufbau gezielt gefördert wird.

- **Sparprodukt-Kampagnen:** Automatisiertes Sparen (Rundup-Feature), Starter-Sparpläne und attraktive Einlagenkonditionen, um den Kontostand schrittweise aufzubauen.
- **Cross-Selling:** Günstige Einstiegskredite, Girokonto-Plus-Pakete mit zusätzlichen Features, Kreditkarten-Upgrades.
- **Gamification in der App:** Spar-Ziele und -Fortschrittsanzeigen, da die App-Nutzung ähnlich hoch ist wie bei Cluster 1 — dieser Kanal ist gut etabliert.

---

**Fazit:** Die drei identifizierten Segmente bilden eine klare strategische Rangfolge ab: Cluster 1 halten, Cluster 2 entwickeln, Cluster 0 zurückgewinnen. Durch den kombinierten Einsatz von CLV, Churn-Status und Kontostands-Analyse können Marketingbudgets effizient auf die Segmente mit dem höchsten ROI (Return on Investment) konzentriert werden — statt auf pauschale Massenkampagnen.